# A3 - Segmentação de Clientes de Varejo

Orientador(es): Joberto Martins e Arthur Kronbauer

Unidade Curricular (UC): Inteligência 

Integrantes:
- Bianca (RA:)
- Edioelson Júnior A. B. Teixeira (1272318423)
- Duilio (RA:)
-

Objetivo: Criar um sistema capaz de segmentar os clientes de um varejo.

Algoritmos utilizados:
- K-Means Clustering
- DBSCAN

Dataset:
- [Mall Customers Dataset](https://www.kaggle.com/datasets/simtoor/mall-customers/data)



## 1. Introdução e objetivo do projeto

(pessoa 1 completar)

- qual é o problema do projeto;
- o que é segmentação de clientes;
- por que isso é útil para empresas de varejo/supermercado;
- qual é o objetivo da aplicação.

## 2. Descrição do dataset

(pessoa 1 completar)

explicar as colunas do dataset:
- CustomerID;
- Gender;
- Age;
- Annual Income (k$);
- Spending Score (1-100).

## Bibliotecas utilizadas
Para o desenvolvimento do projeto foram utilizadas as seguintes bibliotecas Python:
- `pandas`
- `numpy`
- `matplotlib`
- `sklearn`

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

Foram importadas as bibliotecas que vão ser usadas para ler os dados, manipular as informações, gerar gráficos, padronizar as variáveis e aplicar os algoritmos de agrupamento, como K-Means e DBSCAN.

In [ ]:
mall_customers_df = pd.read_csv("../data/Mall_Customers.csv").copy()

print("\n\n---------------------------- MALL CUSTOMERS DATASET ORIGINAL ----------------------------\n")
display(mall_customers_df)

print(f"\n\nO DATASET ORIGINAL POSSUI {mall_customers_df.shape[0]} LINHAS E {mall_customers_df.shape[1]} COLUNAS.")

print("\n\nQUANTIDADE DE VALORES NULLABLE:\n")
display(mall_customers_df.isnull().sum())

print("\n\nINFORMAÇÕES GERAIS:\n")
mall_customers_df.info()



---------------------------- MALL CUSTOMERS DATASET ORIGINAL ----------------------------



,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40
...,...,...,...,...,...
195,196,Female,35,120,79
196,197,Female,45,126,28
197,198,Male,32,126,74
198,199,Male,32,137,18




O DATASET ORIGINAL POSSUI 200 LINHAS E 5 COLUNAS.


QUANTIDADE DE VALORES NULLABLE:



CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64



INFORMAÇÕES GERAIS:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CustomerID              200 non-null    int64 
 1   Gender                  200 non-null    object
 2   Age                     200 non-null    int64 
 3   Annual Income (k$)      200 non-null    int64 
 4   Spending Score (1-100)  200 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 7.9+ KB


## 3. Análise exploratória dos dados

(pessoa 2 completar)

Nesta seção, criar gráficos como:
- quantidade de clientes por gênero;
- distribuição de idade;
- distribuição de renda anual;
- distribuição do spending score;
- relação entre renda anual e spending score.

In [ ]:
X=df[["Annual Income (k$)", "Spending Score (1-100)"]]
X.head()

Para começar o K-Means, foram escolhidas duas colunas principais: renda anual(Annual Income (k$)) e pontuação de gasto(Spending Score (1-100)). Essas informações ajudam a separar clientes de acordo com quanto eles ganham e quanto costumam gastar.


## 4. Pré-processamento dos dados

ja fiz a seleção das variáveis para agrupamento e padronização com StandardScaler

pessoa 3 pode complementar com:
- explicação sobre remoção/ignorância de CustomerID;
- verificação de dados duplicados;
- tratamento da variável Gender, caso o grupo decida usar;
- justificativa das variáveis escolhidas.

In [ ]:
# Padronização dos dados

scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)
X_scaled

In [ ]:
# Transformando os dados padronizados em uma tabela para visualizar melhor

X_scaled_df=pd.DataFrame(
    X_scaled,
    columns=["Renda Anual Padronizada", "Pontuação de Gasto Padronizada"]
)
X_scaled_df.head()

A padronização foi feita com o StandardScaler. Ela deixa as variáveis em uma escala parecida.

Isso é importante porque o K-Means calcula distância entre os pontos. Se uma coluna tiver valores muito maiores que a outra, ela pode acabar pesando mais no resultado.


## 5. Segmentação com K-Means

Nesta etapa foi aplicado o K-Means, que separa os clientes em grupos parecidos de acordo com as variáveis escolhidas.

In [ ]:
# Método do cotovelo para escolher o número de clusters

inertias = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertias, marker="o")
plt.title("Método do Cotovelo")
plt.xlabel("Número de clusters")
plt.ylabel("Inércia")
plt.grid(True)
plt.show()

O método do cotovelo foi usado para ajudar na escolha da quantidade de grupos. A ideia é olhar o ponto em que a queda da inércia começa a ficar menor. Esse ponto ajuda a decidir um valor de K.


In [ ]:
# Cálculo do Silhouette Score para diferentes valores de K

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    print(f"K = {k} | Silhouette Score = {score:.3f}")

O Silhouette Score foi utilizado como métrica para avaliar a qualidade dos agrupamentos. O valor varia de -1 a 1. Quanto mais próximo de 1, melhor a separação entre os clusters.

O valor K = 5 foi escolhido porque apresentou o melhor Silhouette Score entre os valores testados, indicando uma boa separação entre os grupos.

In [ ]:
# Aplicação do K-Means com 5 clusters

kmeans=KMeans(n_clusters=5, random_state=42, n_init=10)

df["Cluster_KMeans"]=kmeans.fit_predict(X_scaled)

df.head()

Após a análise do método do cotovelo e do Silhouette Score, foi escolhido o valor K = 5 para segmentar os clientes em cinco grupos.

In [ ]:
# Quantidade de clientes em cada grupo

df["Cluster_KMeans"].value_counts().sort_index()

Essa contagem mostra quantos clientes ficaram em cada grupo criado pelo K-Means.


In [ ]:
# Média de renda anual e pontuação de gasto por grupo

df.groupby("Cluster_KMeans")[["Annual Income (k$)", "Spending Score (1-100)"]].mean()

As médias de renda anual e pontuação de gasto ajudam a entender o perfil de cada grupo. Com isso, dá para dar um nome mais fácil para cada segmento de cliente.


In [ ]:
# Visualização dos clusters gerados pelo K-Means

plt.figure(figsize=(8, 6))
plt.scatter(
    df["Annual Income (k$)"],
    df["Spending Score (1-100)"],
    c=df["Cluster_KMeans"],
    s=60
)

plt.title("Segmentação de Clientes com K-Means")
plt.xlabel("Renda Anual (k$)")
plt.ylabel("Spending Score (1-100)")
plt.grid(True)
plt.show()

O gráfico mostra a segmentação dos clientes com base na renda anual e na pontuação de gasto. Cada cor representa um grupo diferente identificado pelo algoritmo K-Means.

## 6. Interpretação dos grupos do K-Means

Com base nas médias de renda anual e pontuação de gasto, os grupos foram interpretados da seguinte forma:

Grupo 0: clientes com renda média e gasto médio.
Grupo 1: clientes com alta renda e alto gasto.
Grupo 2: clientes com baixa renda e alto gasto.
Grupo 3: clientes com alta renda e baixo gasto.
Grupo 4: clientes com baixa renda e baixo gasto.

Essa divisão pode ajudar uma empresa de varejo a pensar em campanhas diferentes para cada tipo de cliente.

## 7. Segmentação com DBSCAN
(pessoa 5 completar)

aplicar o algoritmo DBSCAN:
- testar valores de eps;
- testar valores de min_samples;
- criar os clusters;
- identificar ruídos/outliers;
- visualizar os grupos.

## 8. Comparação entre K-Means e DBSCAN

(pessoa 5 completar)
comparar:
- número de grupos encontrados;
- métricas de desempenho;
- facilidade de interpretação;
- presença de ruídos/outliers;
- vantagens e desvantagens de cada algoritmo.

## 9. Aplicação prática para campanhas de marketing
(pessoa 6 completar)

criar uma função simples para:
- receber os dados de um novo cliente;
- classificar o cliente em um grupo;
- mostrar o perfil do grupo;
- sugerir uma campanha de marketing personalizada.

## 10. Conclusão

(grupo completar)

Nesta seção, explicar:
- qual algoritmo funcionou melhor;
- como os grupos podem ajudar uma empresa de varejo;
- limitações do projeto;
- possíveis melhorias futuras.